### To preprocess lidar, remove lidar information from the sides and backwards

crops to number of pixels chosen in a box

In [3]:
import os
import glob
import cv2
import numpy as np
from tqdm import tqdm

# Dataset paths
dataset_path = "/home/edvarsa/pole_detection/lidar_1"
cropped_dataset_path = "/home/edvarsa/pole_detection/lidar_1_cropped"

def create_directory_structure(cropped_dataset_path):
    """Create the directory structure for the cropped dataset"""
    # Create main directories
    os.makedirs(cropped_dataset_path, exist_ok=True)
    
    # Create train and val directories for both images and labels
    for split in ["train", "val"]:
        os.makedirs(os.path.join(cropped_dataset_path, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(cropped_dataset_path, split, "labels"), exist_ok=True)
    
    # Create new data.yaml
    data_yaml_content = f"""train: {cropped_dataset_path}/train/images
val: {cropped_dataset_path}/val/images

nc: 1
names: ['pole']"""
    
    with open(os.path.join(cropped_dataset_path, "data.yaml"), 'w') as f:
        f.write(data_yaml_content)
    
    print("Directory structure created")

def crop_and_resize_images(original_path, cropped_path):
    """Crop images to 256x256 from the center and add black padding above"""
    for split in ["train", "val"]:
        img_dir = os.path.join(original_path, split, "images")
        output_dir = os.path.join(cropped_path, split, "images")
        
        # Get all PNG images
        image_paths = glob.glob(os.path.join(img_dir, "*.png"))
        
        print(f"Processing {len(image_paths)} images in {split} set...")
        
        for img_path in tqdm(image_paths):
            # Read image
            img = cv2.imread(img_path)
            
            if img is None:
                print(f"Could not read image: {img_path}")
                continue
            
            # Original image is 1028x128
            # Calculate center crop coordinates for horizontal crop
            center_x = 1028 // 2
            
            # Calculate crop boundaries to get center portion
            x1 = center_x - 128  # Half of 256
            x2 = center_x + 128
            
            # Crop the center portion horizontally
            cropped_img = img[:, x1:x2]  # Keep full height, crop width
            
            # Create a black 256x256 image
            final_img = np.zeros((256, 256, 3), dtype=np.uint8)
            
            # Place the cropped image at the bottom of the black image
            # Calculate the y-offset to place at bottom
            y_offset = 256 - 128  # 256 (target height) - 128 (original height)
            final_img[y_offset:, :] = cropped_img
            
            # Save final image
            output_path = os.path.join(output_dir, os.path.basename(img_path))
            cv2.imwrite(output_path, final_img)

def process_labels(original_path, cropped_path):
    """Process label files for the cropped images"""
    orig_width = 1028
    orig_height = 128
    new_size = 256
    
    for split in ["train", "val"]:
        label_dir = os.path.join(original_path, split, "labels")
        output_dir = os.path.join(cropped_path, split, "labels")
        
        label_paths = glob.glob(os.path.join(label_dir, "*.txt"))
        print(f"Processing {len(label_paths)} labels in {split} set...")
        
        for label_path in tqdm(label_paths):
            try:
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = parts[0]
                        x_center = float(parts[1])
                        y_center = float(parts[2])
                        width = float(parts[3])
                        height = float(parts[4])
                        
                        # Convert normalized coordinates to absolute
                        abs_x = x_center * orig_width
                        abs_y = y_center * orig_height
                        abs_w = width * orig_width
                        abs_h = height * orig_height
                        
                        # Calculate new coordinates relative to crop
                        crop_x1 = (orig_width - new_size) / 2
                        
                        new_x = abs_x - crop_x1
                        # Adjust y coordinate to account for the black padding at top
                        new_y = abs_y + 128  # Add 128 to shift boxes down
                        
                        # Convert back to normalized coordinates
                        new_x_norm = new_x / new_size
                        new_y_norm = new_y / new_size
                        new_w_norm = abs_w / new_size
                        new_h_norm = abs_h / new_size
                        
                        # Only check if the center of the box is within the cropped region
                        # and if at least 25% of the box is visible
                        if (0 < new_x_norm < 1 and 0.5 < new_y_norm < 1):  # Changed y range to bottom half
                            new_line = f"{class_id} {new_x_norm} {new_y_norm} {new_w_norm} {new_h_norm}\n"
                            new_lines.append(new_line)
                
                # Write processed labels
                output_path = os.path.join(output_dir, os.path.basename(label_path))
                with open(output_path, 'w') as f:
                    f.writelines(new_lines)
                    
            except Exception as e:
                print(f"Error processing {label_path}: {str(e)}")

def main():
    """Main function to create cropped dataset"""
    print("Creating 256x256 center-cropped dataset")
    
    # Step 1: Create directory structure
    create_directory_structure(cropped_dataset_path)
    
    # Step 2: Crop and resize images
    crop_and_resize_images(dataset_path, cropped_dataset_path)
    
    # Step 3: Process label files
    process_labels(dataset_path, cropped_dataset_path)
    
    print(f"Cropped dataset created at: {cropped_dataset_path}")

if __name__ == "__main__":
    main()

Creating 256x256 center-cropped dataset
Directory structure created
Processing 1367 images in train set...


100%|██████████| 1367/1367 [00:10<00:00, 126.62it/s]


Processing 390 images in val set...


100%|██████████| 390/390 [00:03<00:00, 127.82it/s]


Processing 1367 labels in train set...


100%|██████████| 1367/1367 [00:02<00:00, 618.37it/s]


Processing 390 labels in val set...


100%|██████████| 390/390 [00:00<00:00, 639.19it/s]

Cropped dataset created at: /home/edvarsa/pole_detection/lidar_1_cropped
